# 06 - Ablation Study

The core evidence table for the paper:

| Config | What it isolates |
|---|---|
| Baseline | Reference backbone, plain Charbonnier loss |
| + Global Gen. Charbonnier | Fixed p=0.845 from the GLOBAL pooled fit (Notebook 02) |
| + LDMH loss only | Mixture-NLL loss, but NO feature conditioning (FiLM disabled) |
| + LDMH + FiLM | Full mixture conditioning of restoration features |
| Full model | Same as above (this IS the full model in this codebase) |

Each row answers one specific question, per the reviewer-critique roadmap:
does modeling the mixture in the LOSS help; does also conditioning
FEATURES on it help further.

Also evaluates a **hard-pixel (proxy-outlier) metric**: error specifically
on the top-X% highest-residual pixels in each validation image, since that
is where heavy-tail robustness should show up most, mirroring the
outlier-only test from the earlier synthetic Phase-5 experiment.

In [ ]:
import sys, os
sys.path.insert(0, "..")

import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from utils.data import match_pairs, split_pairs, PairedRestorationDataset
from utils.losses import charbonnier_loss, gen_charbonnier_loss
from utils.metrics import psnr_torch, batch_ssim_torch
from models.restoration_net import BaselineNet, DistributionMixtureRestorationNet
from models.ldmh import mixture_gen_gaussian_nll
from pathlib import Path

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
PROJECT_ROOT = Path("..").resolve()

TRAIN_GT_DIR = PROJECT_ROOT / "train" / "train" / "GT"
TRAIN_NOISY_DIR = PROJECT_ROOT / "train" / "train" / "NoisyLR"

MAX_PAIRS = 200
EPOCHS = 3         
BATCH_SIZE = 8
LR = 2e-4
GLOBAL_BETA = 0.845  
HARD_PIXEL_FRAC = 0.05  
os.makedirs("../results/checkpoints", exist_ok=True)


In [3]:
all_pairs = match_pairs(TRAIN_GT_DIR, TRAIN_NOISY_DIR)
if MAX_PAIRS:
    all_pairs = all_pairs[:MAX_PAIRS]
train_pairs, val_pairs = split_pairs(all_pairs, val_frac=0.1, seed=42)
train_ds, val_ds = PairedRestorationDataset(train_pairs), PairedRestorationDataset(val_pairs)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"train: {len(train_pairs)}  val: {len(val_pairs)}")


train: 180  val: 20


### Hard-pixel (proxy-outlier) evaluation metric

In [4]:
@torch.no_grad()
def evaluate_full(model, loader, mode, hard_frac=HARD_PIXEL_FRAC):
    model.eval()
    psnrs, ssims, hard_errs = [], [], []
    for noisy, gt, gt_ds in loader:
        noisy, gt = noisy.to(device), gt.to(device)
        if mode == "baseline":
            pred = model(noisy)
        else:
            pred, _, _, _ = model(noisy)

        psnrs.append(psnr_torch(pred, gt))
        ssims.append(batch_ssim_torch(pred, gt))

        # hard-pixel mask: top hard_frac fraction of |noisy_upsampled - gt| per-sample
        noisy_up = torch.nn.functional.interpolate(noisy, size=gt.shape[-2:], mode="bilinear", align_corners=False)
        orig_err = (noisy_up - gt).abs()
        B = orig_err.shape[0]
        for b in range(B):
            flat = orig_err[b].flatten()
            k = max(1, int(hard_frac * flat.numel()))
            thresh = torch.topk(flat, k).values.min()
            mask = orig_err[b, 0] >= thresh
            err = (pred[b, 0] - gt[b, 0]).abs()[mask].mean().item()
            hard_errs.append(err)

    return float(np.mean(psnrs)), float(np.mean(ssims)), float(np.mean(hard_errs))


### Training function, parameterized by ablation config

In [5]:
def run_config(config_name, model, loss_mode, epochs=EPOCHS, lr=LR, aux_weight=1.0):
    """loss_mode in {'charbonnier', 'gen_charbonnier', 'mixture_only', 'mixture_film'}"""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    is_ldmh = loss_mode in ("mixture_only", "mixture_film")

    for epoch in range(epochs):
        model.train()
        losses = []
        for noisy, gt, gt_ds in train_loader:
            noisy, gt, gt_ds = noisy.to(device), gt.to(device), gt_ds.to(device)
            optimizer.zero_grad()

            if not is_ldmh:
                pred = model(noisy)
                if loss_mode == "charbonnier":
                    loss = charbonnier_loss(pred, gt)
                else:  # gen_charbonnier
                    loss = gen_charbonnier_loss(pred, gt, p=GLOBAL_BETA)
            else:
                pred, mix_weights, beta, scale = model(noisy)
                recon = charbonnier_loss(pred, gt)
                lr_residual = noisy - gt_ds
                mix_nll = mixture_gen_gaussian_nll(lr_residual, mix_weights, beta, scale)
                loss = recon + aux_weight * mix_nll

            loss.backward()
            optimizer.step()
            losses.append(loss.item())

        eval_mode = "baseline" if not is_ldmh else "ldmh"
        psnr, ssim, hard_err = evaluate_full(model, val_loader, mode=eval_mode)
        print(f"[{config_name}] epoch {epoch+1}/{epochs}  loss={np.mean(losses):.4f}  "
              f"val_PSNR={psnr:.3f}  val_SSIM={ssim:.4f}  hard_pixel_err={hard_err:.4f}")

    torch.save(model.state_dict(), f"../results/checkpoints/{config_name}.pt")
    return {"config": config_name, "PSNR": psnr, "SSIM": ssim, "hard_pixel_err": hard_err}


### Run all five ablation configs

In [6]:
results = []

# 1. Baseline: plain Charbonnier
model1 = BaselineNet()
results.append(run_config("1_baseline", model1, loss_mode="charbonnier"))

# 2. Baseline + Global Gen. Charbonnier (fixed p=0.845)
model2 = BaselineNet()
results.append(run_config("2_global_gen_charbonnier", model2, loss_mode="gen_charbonnier"))

# 3. LDMH loss only (mixture NLL), FiLM conditioning DISABLED
model3 = DistributionMixtureRestorationNet(use_film=False)
results.append(run_config("3_ldmh_loss_only", model3, loss_mode="mixture_only"))

# 4. LDMH + FiLM (full mixture conditioning of features)
model4 = DistributionMixtureRestorationNet(use_film=True)
results.append(run_config("4_ldmh_plus_film", model4, loss_mode="mixture_film"))


[1_baseline] epoch 1/3  loss=0.0615  val_PSNR=24.749  val_SSIM=0.5527  hard_pixel_err=0.1531
[1_baseline] epoch 2/3  loss=0.0478  val_PSNR=25.651  val_SSIM=0.5886  hard_pixel_err=0.1426
[1_baseline] epoch 3/3  loss=0.0438  val_PSNR=26.112  val_SSIM=0.6073  hard_pixel_err=0.1338
[2_global_gen_charbonnier] epoch 1/3  loss=0.0765  val_PSNR=25.383  val_SSIM=0.5674  hard_pixel_err=0.1431
[2_global_gen_charbonnier] epoch 2/3  loss=0.0675  val_PSNR=26.188  val_SSIM=0.6100  hard_pixel_err=0.1306
[2_global_gen_charbonnier] epoch 3/3  loss=0.0642  val_PSNR=26.477  val_SSIM=0.6255  hard_pixel_err=0.1231
[3_ldmh_loss_only] epoch 1/3  loss=0.4283  val_PSNR=25.004  val_SSIM=0.5688  hard_pixel_err=0.1494
[3_ldmh_loss_only] epoch 2/3  loss=0.3363  val_PSNR=25.730  val_SSIM=0.5935  hard_pixel_err=0.1411
[3_ldmh_loss_only] epoch 3/3  loss=0.3113  val_PSNR=26.105  val_SSIM=0.6091  hard_pixel_err=0.1339
[4_ldmh_plus_film] epoch 1/3  loss=0.4322  val_PSNR=25.291  val_SSIM=0.5773  hard_pixel_err=0.1456
[4_l

In [7]:
import pandas as pd
results_df = pd.DataFrame(results)
results_df.to_csv("../results/06_ablation_table.csv", index=False)
print(results_df.to_string(index=False))


                  config      PSNR     SSIM  hard_pixel_err
              1_baseline 26.111686 0.607326        0.133765
2_global_gen_charbonnier 26.476780 0.625529        0.123102
        3_ldmh_loss_only 26.104947 0.609095        0.133853
        4_ldmh_plus_film 26.368247 0.620447        0.126403


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(16,4.5))
for ax, metric, title in zip(axes, ["PSNR", "SSIM", "hard_pixel_err"],
                              ["Validation PSNR (higher better)", "Validation SSIM (higher better)",
                               "Hard-pixel error (lower better)"]):
    ax.bar(results_df["config"], results_df[metric])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("../results/06_ablation_bars.png", dpi=150)
plt.show()


C:\Users\DELL\AppData\Local\Temp\ipykernel_31528\1077681268.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
